# 01 — Exploratory Data Analysis, Cleaning & Tourism Integration

- **Owner:** Maxime Laurent (`@maximelrnt` — Team Leader)
- **Course Rubric:** Task 1 — Data Exploration `[2 pts]` & Preprocessing for External Bonus `[+1 pt]`
- **Datasets:** Global Shark Attack File (`projet/data/attacks.csv`) & World Bank International Tourism (`projet/data/tourism_arrivals.csv`)

In [1]:
# Import libraries and shared project data loader
import os
import sys
import warnings
import pandas as pd
import numpy as np
import plotly.express as px

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import (
    load_shark_data,
    load_tourism_data,
    summarize_dataset,
    clean_shark_data,
    clean_and_merge_tourism,
)

warnings.filterwarnings('ignore')
SHARK_PATH = '../data/attacks.csv'
TOURISM_PATH = '../data/tourism_arrivals.csv'

## 1. Human-Readable Description of the Datasets

1. **Primary Multidimensional Dataset — Global Shark Attack File (GSAF):**
   Maintained by the Shark Research Institute, this dataset logs documented shark-human encounters worldwide across three dimensions:
   - **Temporal:** `Date`, `Year`, `Time` (incident chronology).
   - **Spatial:** `Country`, `Area` (state/province), `Location` (specific beach or reef).
   - **Analytical:** `Type` (Unprovoked, Provoked, Boating, etc.), `Activity`, `Species`, victim `Sex`/`Age`, `Injury`, and `Fatal (Y/N)`.
2. **External Bonus Dataset — World Bank International Tourism (`ST.INT.ARVL`):**
   Official World Bank indicator tracking annual inbound overnight international tourist arrivals across 266 countries and regions from **1995 to 2020**.
   Joining both datasets on `(Country, Year)` enables distinguishing rising shark encounters caused by growing human beach exposure from intrinsic coastal risk.

In [2]:
# Load raw datasets and compute full profiling summaries
raw_sharks = load_shark_data(SHARK_PATH)
raw_tourism = load_tourism_data(TOURISM_PATH)

shark_summary = summarize_dataset(raw_sharks, 'Global Shark Attack (Raw)')
tourism_summary = summarize_dataset(raw_tourism, 'World Bank Tourism ST.INT.ARVL (Raw)')

print(f"Raw Shark Attacks: {shark_summary['n_rows']} rows x {shark_summary['n_cols']} columns")
print(f"Raw World Bank Tourism: {tourism_summary['n_rows']} rows x {tourism_summary['n_cols']} columns")

# Build structured profiling table for the primary dataset
profile_df = pd.DataFrame({
    'Column': shark_summary['columns'],
    'Data_Type': [shark_summary['dtypes'][c] for c in shark_summary['columns']],
    'Missing_Count': [shark_summary['missing_counts'][c] for c in shark_summary['columns']],
    'Missing_Pct': [shark_summary['missing_pct'][c] for c in shark_summary['columns']],
})
display(profile_df)
print('Numeric column ranges (Raw Sharks):', shark_summary['numeric_ranges'])

Raw Shark Attacks: 25723 rows x 24 columns
Raw World Bank Tourism: 266 rows x 30 columns


,Column,Data_Type,Missing_Count,Missing_Pct
0,Case Number,str,17021,66.17
1,Date,str,19421,75.50
2,Year,float64,19423,75.51
3,Type,str,19425,75.52
4,Country,str,19471,75.69
5,Area,str,19876,77.27
6,Location,str,19961,77.60
7,Activity,str,19965,77.62
8,Name,str,19631,76.32
9,Sex,str,19986,77.70


Numeric column ranges (Raw Sharks): {'Year': {'min': 0.0, 'max': 2018.0, 'mean': 1927.27, 'median': 1977.0}, 'original order': {'min': 2.0, 'max': 6310.0, 'mean': 3156.0, 'median': 3156.0}}


In [3]:
# Visualize missingness percentage per column in the raw GSAF file
fig_missing = px.bar(
    profile_df.sort_values('Missing_Pct', ascending=True),
    x='Missing_Pct',
    y='Column',
    orientation='h',
    color='Missing_Pct',
    color_continuous_scale='Blues',
    labels={'Missing_Pct': 'Missing Values (%)', 'Column': 'Raw Column'},
    title='Raw GSAF Dataset: Missing Values per Column (includes trailing blank export rows)',
)
fig_missing.update_layout(template='plotly_white', height=560)
fig_missing.show()

## 2. Documented Cleaning & Feature Engineering Decisions

Every transformation applied inside `clean_shark_data()` is documented below:
- **Trailing blank rows dropped:** Raw `attacks.csv` contains 25,723 lines, of which over 17,000 are completely blank spreadsheet padding rows. All rows with empty `Case Number` are removed.
- **Administrative columns dropped:** `href`, `href formula`, `pdf`, `Case Number.1`, `Case Number.2`, `original order`, `Unnamed: 22`, `Unnamed: 23`, and `Investigator or Source` are internal file references/export artifacts that add no analytical information.
- **Modern temporal filter (`1960 <= Year <= 2023`):** Historical records before 1960 suffer from heavy under-reporting, missing dates, and uncertain species attribution. Keeping 1960-2018 preserves **4,033 reliable modern incidents**.
- **Country harmonization (`Country_clean`):** Stripped whitespace and trailing `?` characters, then mapped variants (`USA` -> `United States`, `ENGLAND`/`SCOTLAND` -> `United Kingdom`, `BAHAMAS` -> `Bahamas, The`, `EGYPT` -> `Egypt, Arab Rep.`) to align cleanly with World Bank names.
- **Feature categorization:** Grouped noisy free-text `Activity` into 7 clean categories (`Activity_group`), `Species` into 8 groups (`Species_group`), `Time` into 4 periods (`Time_of_day`), `Fatal (Y/N)` into `Y`/`N`/`Unknown` (`Fatal_clean`), and derived hemisphere-aware `Season` from `Month` and `Country_clean`.

In [4]:
# Apply cleaning pipeline and inspect post-cleaning profile
clean_sharks = clean_shark_data(raw_sharks)
clean_summary = summarize_dataset(clean_sharks, 'Global Shark Attack (Cleaned 1960-2018)')

print(f"Cleaned Shark Attacks: {clean_summary['n_rows']} rows x {clean_summary['n_cols']} columns")
print('Numeric ranges after cleaning:', clean_summary['numeric_ranges'])
display(clean_sharks[['Year', 'Month', 'Season', 'Country_clean', 'Area_clean', 'Activity_group', 'Species_group', 'Time_of_day', 'Fatal_clean']].head(10))

Cleaned Shark Attacks: 4033 rows x 26 columns
Numeric ranges after cleaning: {'Year': {'min': 1960.0, 'max': 2018.0, 'mean': 1994.79, 'median': 2000.0}, 'Month': {'min': 1.0, 'max': 12.0, 'mean': 6.57, 'median': 7.0}, 'Age_clean': {'min': 1.0, 'max': 87.0, 'mean': 28.29, 'median': 25.0}}


,Year,Month,Season,Country_clean,Area_clean,Activity_group,Species_group,Time_of_day,Fatal_clean
0,2018,6.0,Summer,United States,California,Other,White Shark,Evening/Night (18-06),N
1,2018,6.0,Summer,United States,Georgia,Wading,Unidentified,Afternoon (12-18),N
2,2018,6.0,Summer,United States,Hawaii,Surfing,Unidentified,Morning (06-12),N
3,2018,6.0,Winter,Australia,New South Wales,Surfing,Other Named Shark,Unknown,N
4,2018,6.0,Summer,Mexico,Colima,Diving/Snorkeling,Tiger Shark,Unknown,N
5,2018,6.0,Winter,Australia,New South Wales,Surfing,Unidentified,Unknown,N
6,2018,6.0,Winter,Brazil,Pernambuco,Swimming,Tiger Shark,Afternoon (12-18),Y
7,2018,5.0,Spring,United States,Florida,Fishing,Reef/Nurse Shark,Unknown,N
8,2018,5.0,Spring,United States,Florida,Wading,Bull Shark,Afternoon (12-18),N
9,2018,5.0,Spring,United States,Florida,Wading,Unidentified,Afternoon (12-18),N


In [5]:
# Reshape World Bank tourism data and verify (Country_clean, Year) join coverage
merged_tourism = clean_and_merge_tourism(clean_sharks, raw_tourism)
print(f'Merged country-year records (1995-2018): {len(merged_tourism)} rows across {merged_tourism["Country_clean"].nunique()} countries')
print(f'Overlapping year span: {merged_tourism["Year"].min()} to {merged_tourism["Year"].max()}')
display(merged_tourism.head(10))

# Check which top post-1995 countries had no World Bank tourism entry (e.g., overseas territories like Reunion)
sharks_95 = clean_sharks[clean_sharks['Year'] >= 1995]
matched_countries = set(merged_tourism['Country_clean'].unique())
unmatched = sharks_95[~sharks_95['Country_clean'].isin(matched_countries)]['Country_clean'].value_counts().head(8)
print('\nTop unmatched territories/entries in World Bank ST.INT.ARVL (documented limitation):')
print(unmatched)

Merged country-year records (1995-2018): 355 rows across 65 countries


Overlapping year span: 1995 to 2018


,Country_clean,Country Code,Year,Arrivals,annual_attacks,fatal_attacks
0,Aruba,ABW,2015,1832000.0,1,1
1,Australia,AUS,1995,3726000.0,8,1
2,Australia,AUS,1996,4165000.0,12,1
3,Australia,AUS,1997,4318000.0,10,1
4,Australia,AUS,1998,4167000.0,5,1
5,Australia,AUS,1999,4459000.0,3,1
6,Australia,AUS,2000,4931000.0,18,3
7,Australia,AUS,2001,4856000.0,13,0
8,Australia,AUS,2002,4841000.0,13,2
9,Australia,AUS,2003,4746000.0,10,1



Top unmatched territories/entries in World Bank ST.INT.ARVL (documented limitation):
Country_clean
Reunion                                  40
Turks & Caicos                            5
Atlantic Ocean                            4
Taiwan                                    4
Okinawa                                   4
Columbia                                  3
Madagascar                                3
St Helena, British Overseas Territory     2
Name: count, dtype: int64


## 3. EDA Key Takeaways

- Cleaning reduced 25,723 raw rows (including blank CSV rows and pre-1960 entries) to **4,033 clean modern incidents** across 1960-2018.
- Overall fatality rate across the cleaned dataset is **12.3%** (495 fatal outcomes).
- World Bank tourism integration successfully matches **65 countries** over **1995-2018** (355 country-year pairs), including all top affected nations (`United States`, `Australia`, `South Africa`, `Brazil`, `Bahamas, The`, `New Zealand`, `New Caledonia`, `Mexico`). Overseas departments such as `Reunion` are not listed separately in World Bank national accounts and are transparently documented.